# Day 9: Decision Trees & Random Forests

## Task 1: Wine Dataset - Decision Tree vs Random Forest

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
wine = load_wine()

X = wine.data
y = wine.target
feature_names = wine.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Decision Tree with max_depth=None (Full Tree)

In [ ]:
dt_full = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

dt_full.fit(X_train, y_train)

train_pred_full = dt_full.predict(X_train)
test_pred_full = dt_full.predict(X_test)

train_acc_full = accuracy_score(y_train, train_pred_full)
test_acc_full = accuracy_score(y_test, test_pred_full)

print("Decision Tree (max_depth=None)")
print("Train Accuracy:", train_acc_full)
print("Test Accuracy:", test_acc_full)
print("Gap:", train_acc_full - test_acc_full)

### Decision Tree with max_depth=3 (Pruned Tree)

In [ ]:
dt_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

dt_depth3.fit(X_train, y_train)

train_pred_depth3 = dt_depth3.predict(X_train)
test_pred_depth3 = dt_depth3.predict(X_test)

train_acc_depth3 = accuracy_score(y_train, train_pred_depth3)
test_acc_depth3 = accuracy_score(y_test, test_pred_depth3)

print("Decision Tree (max_depth=3)")
print("Train Accuracy:", train_acc_depth3)
print("Test Accuracy:", test_acc_depth3)
print("Gap:", train_acc_depth3 - test_acc_depth3)

### Random Forest with Different Number of Trees

In [ ]:
tree_counts = [50, 100, 200]

for n in tree_counts:
    rf = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )

    rf.fit(X_train, y_train)

    train_acc = rf.score(X_train, y_train)
    test_acc = rf.score(X_test, y_test)

    print(f"Random Forest ({n} Trees)")
    print("Train Accuracy:", train_acc)
    print("Test Accuracy:", test_acc)

### Feature Importance

In [ ]:
rf_final = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_final.fit(X_train, y_train)

importances = rf_final.feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

top10 = importance_df.head(10)

print("Top 10 Features")
print(top10)

plt.figure(figsize=(10, 6))

plt.barh(
    top10["Feature"][::-1],
    top10["Importance"][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importances")
plt.tight_layout()
plt.show()

### Out-of-Bag (OOB) Score

In [ ]:
rf_oob = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    random_state=42
)

rf_oob.fit(X_train, y_train)

print("OOB Score:", rf_oob.oob_score_)
print("Test Accuracy:", rf_oob.score(X_test, y_test))

## Task 2: Breast Cancer Dataset - Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
cancer = load_breast_cancer()

X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Baseline Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

baseline_accuracy = rf.score(X_test, y_test)

print("Test Accuracy Before Tuning:")
print(baseline_accuracy)

### GridSearchCV for Hyperparameter Tuning

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7, None],
    'min_samples_leaf': [1, 5, 10],
    'max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("Best Cross Validation Score:")
print(grid.best_score_)

### Evaluate Tuned Model

In [ ]:
best_model = grid.best_estimator_

tuned_accuracy = best_model.score(X_test, y_test)

print("Test Accuracy After Tuning:")
print(tuned_accuracy)

improvement = tuned_accuracy - baseline_accuracy

print("Improvement in Test Accuracy:")
print(improvement)